# 01 — Preparar perfil de voz XTTS

Notebook independiente para crear una voz propia reutilizable por el asistente. Usa XTTS-v2 de forma local en Colab: procesa una grabación autorizada y guarda los latentes de condicionamiento en Google Drive. **No se entrena un modelo nuevo** y no se llama a una API externa de inferencia.

El perfil es sensible: guárdalo en Drive privado. No lo subas a GitHub ni compartas el notebook con acceso a ese archivo.


## 1. Activar GPU y montar Drive

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
import torch, os
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE)
assert DEVICE == 'cuda', 'En Colab, selecciona Entorno de ejecución → Cambiar tipo de entorno → GPU.'


## 2. Instalar y cargar XTTS-v2

In [ ]:
!pip -q install TTS
from TTS.api import TTS
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(DEVICE)
model = tts.synthesizer.tts_model
print('XTTS cargado')


## 3. Subir una grabación de referencia
Usa tu propia voz. Graba en un lugar silencioso, con voz natural y sin música. Una referencia limpia facilita que el perfil recoja mejor el timbre y la forma de hablar.

In [ ]:
uploaded = files.upload()
assert uploaded, 'Sube un audio de referencia.'
reference_path = next(iter(uploaded))
print('Audio seleccionado:', reference_path)


## 4. Crear el perfil de voz
XTTS calcula dos representaciones internas a partir del audio de referencia. Guardarlas evita recalcularlas en cada sesión; el modelo XTTS sigue siendo necesario para sintetizar.

In [ ]:
with torch.inference_mode():
    gpt_cond_latent, speaker_embedding = model.get_conditioning_latents(
        audio_path=[reference_path],
        gpt_cond_len=6,
        max_ref_length=30,
        sound_norm_refs=False
    )
print('Perfil calculado:', tuple(gpt_cond_latent.shape), tuple(speaker_embedding.shape))


## 5. Guardar perfil en Drive privado

In [ ]:
profile_dir = '/content/drive/MyDrive/lab4'
os.makedirs(profile_dir, exist_ok=True)
profile_path = os.path.join(profile_dir, 'voice_profile_winz.pt')
torch.save({
    'gpt_cond_latent': gpt_cond_latent.detach().cpu(),
    'speaker_embedding': speaker_embedding.detach().cpu(),
    'model': 'XTTS-v2',
    'language': 'es'
}, profile_path)
print('Perfil guardado en Drive:', profile_path)


## 6. Probar una frase nueva
La frase de prueba debe ser distinta de lo dicho en la referencia. La generación usa los latentes guardados y la función de inferencia del modelo.

In [ ]:
profile = torch.load(profile_path, map_location=DEVICE, weights_only=True)
text = 'Hola, esta es una prueba de síntesis con mi perfil de voz.'
with torch.inference_mode():
    result = model.inference(
        text=text, language='es',
        gpt_cond_latent=profile['gpt_cond_latent'].to(DEVICE),
        speaker_embedding=profile['speaker_embedding'].to(DEVICE),
        enable_text_splitting=True
    )
from IPython.display import Audio, display
display(Audio(result['wav'], rate=24000))


## 7. Qué reutiliza el notebook de compras
En `Lab4_Asistente_Compras_por_Voz_MVP.ipynb`, monta Drive, carga `voice_profile_winz.pt` y pasa sus dos tensores a `model.inference()` para producir el audio de cada respuesta. Si quieres reemplazar la voz, vuelve a ejecutar este notebook y guarda el nuevo perfil con el mismo nombre.

**Seguridad:** el perfil de voz permite generar audio parecido a tu voz; mantén el archivo privado. El notebook de compras contiene solo la ruta, no el perfil.